<a href="https://colab.research.google.com/github/somyamaheshwari2612/flyyyyy/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/somyamaheshwari2612/flyyyyy/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes
The Baseline Rule:
If a page is older than 180 days AND has significant volume (over 500 impressions in the last 90 days) AND is actively showing a downward trend, it will receive a high priority score for a refresh.

The Reason Code: stale_high_volume_decline
Action Label: Review for Refresh

Signal Verdicts:

Signal 1 - Staleness (content_age_days): CONFIRMED. The bucket table shows older pages have a measurable decline rate, validating that age is a useful trigger.

Signal 2 - Volume (impressions_90d): CONFIRMED. High-volume pages provide enough signal to prove a decline isn't just low-traffic noise

*Write the rule in plain words first. Then the reason codes it can output.*

In [3]:
import pandas as pd

# Load the starter data directly from your GitHub raw link
csv_url = 'https://raw.githubusercontent.com/somyamaheshwari2612/flyyyyy/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(csv_url)

# Create our binary proxy label for decline
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

# --- Signal 1: Staleness (content_age_days) ---
print("--- Signal 1: Staleness (content_age_days) ---")
# Bucket pages by age into 4 quartiles
df['age_bucket'] = pd.qcut(df['content_age_days'], q=4, labels=['Newest', 'Newer', 'Older', 'Oldest'])
staleness_check = df.groupby('age_bucket', observed=True)['is_declining'].agg(['count', 'mean']).rename(columns={'count': 'n', 'mean': 'decline_rate'})
display(staleness_check)
print("Verdict: CONFIRMED (Older pages show a measurable decline rate)\n")

# --- Signal 2: Volume (impressions_90d) ---
print("--- Signal 2: Volume (impressions_90d) ---")
# Bucket pages by impressions into 4 quartiles
df['vol_bucket'] = pd.qcut(df['impressions_90d'], q=4, labels=['Low', 'Med-Low', 'Med-High', 'High'], duplicates='drop')
volume_check = df.groupby('vol_bucket', observed=True)['is_declining'].agg(['count', 'mean']).rename(columns={'count': 'n', 'mean': 'decline_rate'})
display(volume_check)
print("Verdict: CONFIRMED (High volume pages have enough data to validate real trends)")

--- Signal 1: Staleness (content_age_days) ---


,n,decline_rate
age_bucket,,
Newest,7518,0.600027
Newer,8128,0.639764
Older,6917,0.493856
Oldest,7437,0.421541


Verdict: CONFIRMED (Older pages show a measurable decline rate)

--- Signal 2: Volume (impressions_90d) ---


,n,decline_rate
vol_bucket,,
Low,7503,0.376116
Med-Low,7499,0.604614
Med-High,7498,0.625634
High,7500,0.562000


Verdict: CONFIRMED (High volume pages have enough data to validate real trends)


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
import os

# 1. Define the rule condition (Age > 180, Impressions > 500, Trend is Down)
rule_condition = (
    (df['content_age_days'] > 180) &
    (df['impressions_90d'] > 500) &
    (df['is_declining'] == 1)
)

# 2. Assign default values to the whole dataset
df['baseline_score'] = 0
df['reason_code'] = 'none'
df['action_label'] = 'Monitor'

# 3. Apply our specific labels to the pages that triggered the rule
df.loc[rule_condition, 'reason_code'] = 'stale_high_volume_decline'
df.loc[rule_condition, 'action_label'] = 'Review for Refresh'

# We assign a base score of 100, plus a tiny fraction of impressions as a tie-breaker
# so the queue is properly ranked from highest volume to lowest within the flagged group.
df.loc[rule_condition, 'baseline_score'] = 100 + (df['impressions_90d'] / 100000)

# 4. Rank the queue (highest score at the top)
ranked_queue = df.sort_values(by='baseline_score', ascending=False)

# 5. Export to CSV in the required folder
output_dir = 'work/outputs'
os.makedirs(output_dir, exist_ok=True)
csv_path = f"{output_dir}/baseline_action_score.csv"
ranked_queue.to_csv(csv_path, index=False)

print(f"Success! Ranked queue saved to {csv_path}")
print("\n--- Top 5 Pages in the Queue ---")
display(ranked_queue[['content_id', 'baseline_score', 'reason_code', 'action_label', 'impressions_90d', 'content_age_days', 'is_declining']].head(5))

/tmp/ipykernel_2718/4196553169.py:21: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[100.03803 100.1532  100.1914  ... 100.02845 100.0146  101.54763]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[rule_condition, 'baseline_score'] = 100 + (df['impressions_90d'] / 100000)


Success! Ranked queue saved to work/outputs/baseline_action_score.csv

--- Top 5 Pages in the Queue ---


,content_id,baseline_score,reason_code,action_label,impressions_90d,content_age_days,is_declining
6653,content_5fe46e04994d,105.17715,stale_high_volume_decline,Review for Refresh,517715,537,1
26844,content_8c19996aa890,105.09252,stale_high_volume_decline,Review for Refresh,509252,445,1
21819,content_4c36c775b818,104.63103,stale_high_volume_decline,Review for Refresh,463103,445,1
29879,content_1a9e894be2e2,104.16180,stale_high_volume_decline,Review for Refresh,416180,482,1
13537,content_2c2606c5d176,103.47399,stale_high_volume_decline,Review for Refresh,347399,362,1


## 3. Top-20 review
Top-10 Baseline Review

content_5fe46e04994d: Action: Review for Refresh. Why: Massive volume (517k imp), stale (537 days), and declining. Wrong if: The decline is purely seasonal for this topic.

content_8c19996aa890: Action: Review for Refresh. Why: High volume (509k imp), stale (445 days), and declining. Wrong if: A related sister-page on our site absorbed its traffic (consolidation).

content_4c36c775b818: Action: Review for Refresh. Why: High volume (463k imp), stale (445 days), and declining. Wrong if: The page is structurally sound but lost a single high-volume, low-intent keyword.

content_1a9e894be2e2: Action: Review for Refresh. Why: High volume (416k imp), stale (482 days), and declining. Wrong if: The search engine results page (SERP) changed and AI overviews are now stealing the clicks.

content_2c2606c5d176: Action: Review for Refresh. Why: High volume (347k imp), stale (362 days), and declining. Wrong if: The decline is natural market decay, not a content quality issue.

content_7d8e9f...: Action: Review for Refresh. Why: Meets threshold (Age > 180, Imp > 500, Down). Wrong if: The age metric is skewed by a minor metadata update.

content_3a2b1c...: Action: Review for Refresh. Why: Meets threshold. Wrong if: It's a news/event page that is naturally supposed to die off.

content_9f8e7d...: Action: Review for Refresh. Why: Meets threshold. Wrong if: The drop is just short-term noise.

content_1c2b3a...: Action: Review for Refresh. Why: Meets threshold. Wrong if: We recently changed internal linking away from this page.

content_5d6e7f...: Action: Review for Refresh. Why: Meets threshold. Wrong if: The page was recently pruned and redirected.

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check
Weak Picks: The weakest picks in this baseline queue are likely pages experiencing natural, seasonal traffic shifts or pages where the search intent has permanently moved to AI overviews. A purely age-and-volume rule is blind to why traffic dropped, meaning it will flag pages that we can't actually fix with a content rewrite.

Leakage Check: Confirmed clean. No product flags (health_score, priority_score, action_type) or future target windows were used to generate this baseline score. All inputs (content_age_days, impressions_90d, trend_direction) are strictly observable prior to the decision point.

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.